# American Community Survey API (2020-2024, 5 year estimate)

## Author: Yi
## Group: Yi Wang, An Nisa Astuti

### Notebook Description: 
This notebook pulls tract-level ACS 5-year estimates for Cook County (state 17, county 031) from the Census ACS API (2020–2024), then spatially assigns each tract to a Chicago community area via a centroid-within join and aggregates tract counts into community-area SES rates. The result was saved to **acs_community_area_ses_2020_2024.csv** Then, we merged **311_metrics.csv** with **acs_community_area_ses_2020_2024.csv** to prepare a merge ready table with school profile data called **master_acs_311_ca.csv**.

### AI Statement 
- I used ChatGPT to learn the geopandas library for the geospatial steps in this notebook, and learned the logic of CRS transformations, computing tract centroids, and performing spatial joins with geopandas.sjoin to map ACS tracts to Chicago geographies. 

In [24]:
import requests
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import time
import os

In [25]:
os.environ["CENSUS_API_KEY"] = "e81bd440211b1791abce5c3b7b55f12343696a58"

In [26]:
ACS_YEAR = 2024  
STATE_FIPS = "17"  # IL
COOK_FIPS = "031"  # Cook County
CENSUS_API_KEY = os.getenv("CENSUS_API_KEY")  

# Community area boundaries 
COMMUNITY_AREAS_URL = "https://raw.githubusercontent.com/thisisdaryn/data/master/geo/chicago/Comm_Areas.geojson"

# tract shapefile zip for Illinois (2024)
TRACT_ZIP_URL = "https://www2.census.gov/geo/tiger/TIGER2024/TRACT/tl_2024_17_tract.zip"
TRACT_ZIP_LOCAL = "tl_2024_17_tract.zip"


PROJECTED_CRS = "EPSG:3435"  # Illinois East 

# ACS variables from the data filter on acs website
ACS_VARS = {

    # Tenure
    "B25003_001E": "occ_hu_tenure_total",
    "B25003_003E": "renter_occupied",

    # Vacancy
    "B25002_001E": "hu_total",
    "B25002_003E": "hu_vacant",

    # Severe rent burden (50%+)
    "B25070_010E": "rent_50_plus",

    # Per-capita income
    "B19301_001E": "per_capita_income",

    # Public assistance cash income
    "B19057_001E": "hh_total_pa",
    "B19057_002E": "hh_public_assist",

    # No vehicle
    "B08201_001E": "hh_total_veh",
    "B08201_002E": "hh_no_vehicle",

    # Inequality
    "B19083_001E": "gini",

    # Population denominators
    "B01003_001E": "total_population",

    # Poverty
    "B17001_001E": "poverty_denom",
    "B17001_002E": "below_poverty",

    # Income
    "B19013_001E": "median_household_income",

    # Employment
    "B23025_003E": "labor_force",
    "B23025_005E": "unemployed",

    # Educational attainment (25+): for BA+ and <HS
    "B15003_001E": "edu_denom",
    "B15003_002E": "edu_no_schooling",
    "B15003_003E": "edu_nursery",
    "B15003_004E": "edu_k",
    "B15003_005E": "edu_g1",
    "B15003_006E": "edu_g2",
    "B15003_007E": "edu_g3",
    "B15003_008E": "edu_g4",
    "B15003_009E": "edu_g5",
    "B15003_010E": "edu_g6",
    "B15003_011E": "edu_g7",
    "B15003_012E": "edu_g8",
    "B15003_013E": "edu_g9",
    "B15003_014E": "edu_g10",
    "B15003_015E": "edu_g11",
    "B15003_016E": "edu_g12_no_diploma",
    "B15003_022E": "edu_bachelors",
    "B15003_023E": "edu_masters",
    "B15003_024E": "edu_professional",
    "B15003_025E": "edu_doctorate",

    # SNAP (public assistance proxy)
    "B22010_001E": "hh_total",
    "B22010_002E": "hh_snap",

    # Rent burden (>=30% gross rent / household income)
    # total + four >=30% bins + not computed
    "B25070_001E": "rent_burden_total",
    "B25070_007E": "rent_30_34",

    "B25070_008E": "rent_35_39",
    "B25070_009E": "rent_40_49",
    "B25070_010E": "rent_50_plus",
    "B25070_011E": "rent_not_computed",

    # Crowding (>1.0 persons/room): occupied housing units
    "B25014_001E": "occ_hu_total",
    # owner crowded bins (>1.0): 1.01–1.50, 1.51–2.00, 2.01+
    "B25014_005E": "own_crowd_101_150",
    "B25014_006E": "own_crowd_151_200",
    "B25014_007E": "own_crowd_201_plus",
    # renter crowded bins (>1.0): 1.01–1.50, 1.51–2.00, 2.01+
    "B25014_011E": "rent_crowd_101_150",
    "B25014_012E": "rent_crowd_151_200",
    "B25014_013E": "rent_crowd_201_plus",

    # Race/ethnicity 
    "B03002_001E": "race_total",
    "B03002_003E": "white_non_hispanic",
    "B03002_004E": "black_non_hispanic",
    "B03002_012E": "hispanic",
}

VAR_NAMES = list(ACS_VARS.keys())


In [27]:
# Pull ACS tract-level data for Cook County
def pull_acs_tracts(api_key: str, year: int, variables: list) -> pd.DataFrame:
    base_url = f"https://api.census.gov/data/{year}/acs/acs5"

    chunk_size = 45  
    var_chunks = [variables[i:i + chunk_size] for i in range(0, len(variables), chunk_size)]

    all_chunks = []
    for chunk in var_chunks:
        params = {
            "get": "NAME," + ",".join(chunk),
            "for": "tract:*",
            "in": f"state:{STATE_FIPS} county:{COOK_FIPS}",
            "key": api_key,
        }
        print(f"  Pulling {len(chunk)} vars...")
        resp = requests.get(base_url, params=params, timeout=60)
        resp.raise_for_status()

        data = resp.json()
        df = pd.DataFrame(data[1:], columns=data[0])

        # GEOID for tracts (11-digit): state+county+tract
        df["geoid"] = df["state"] + df["county"] + df["tract"]

        df = df.drop(columns=["NAME", "state", "county", "tract"])
        all_chunks.append(df)
        time.sleep(0.4)

    # merge chunks on geoid
    out = all_chunks[0]
    for d in all_chunks[1:]:
        out = out.merge(d, on="geoid", how="inner")

    out = out.rename(columns=ACS_VARS)

    for c in out.columns:
        if c != "geoid":
            out[c] = pd.to_numeric(out[c], errors="coerce")
            out[c] = out[c].replace(-666666666, pd.NA)

    print(f"  Pulled {len(out)} Cook County tracts.")
    return out

In [28]:
# Load boundaries (community areas + tracts)
def load_community_area_boundaries() -> gpd.GeoDataFrame:
    print("  Downloading Chicago community area boundaries...")
    gdf = gpd.read_file(COMMUNITY_AREAS_URL)

    # area number, community, geometry
    if "area_numbe" not in gdf.columns or "community" not in gdf.columns:
        raise KeyError(f"Unexpected CA columns. Got: {gdf.columns.tolist()}")

    gdf["community_area"] = pd.to_numeric(gdf["area_numbe"], errors="coerce").astype("Int64")
    gdf = gdf.rename(columns={"community": "community_name"})
    gdf = gdf[["community_area", "community_name", "geometry"]].set_crs("EPSG:4326", allow_override=True)

    # drop null IDs 
    gdf = gdf.dropna(subset=["community_area"]).copy()
    print(f"  Loaded {len(gdf)} community areas.")
    return gdf


def ensure_tract_zip():
    if os.path.exists(TRACT_ZIP_LOCAL):
        return
    r = requests.get(TRACT_ZIP_URL, timeout=180)
    r.raise_for_status()
    with open(TRACT_ZIP_LOCAL, "wb") as f:
        f.write(r.content)


def load_tract_boundaries() -> gpd.GeoDataFrame:
    ensure_tract_zip()
    tracts = gpd.read_file(TRACT_ZIP_LOCAL)
    tracts = tracts[tracts["COUNTYFP"] == COOK_FIPS].copy()
    tracts["geoid"] = tracts["GEOID"]  # 11-digit tract GEOID (state+county+tract)
    tracts = tracts[["geoid", "geometry"]].set_crs("EPSG:4326", allow_override=True)
    print(f"  Loaded {len(tracts)} Cook County tracts.")
    return tracts


def build_crosswalk(tract_gdf: gpd.GeoDataFrame, ca_gdf: gpd.GeoDataFrame) -> pd.DataFrame:
    """
    Mapping centroid-based tract to community area assignment.
    Fast and usually fine for CA-level aggregation.
    """
    # compute centroids in projected CRS to avoid geographic centroid issues
    t = tract_gdf.to_crs(PROJECTED_CRS).copy()
    ca = ca_gdf.to_crs(PROJECTED_CRS).copy()
    t["geometry"] = t.geometry.centroid

    joined = gpd.sjoin(
        t[["geoid", "geometry"]],
        ca[["community_area", "community_name", "geometry"]],
        how="inner",
        predicate="within",
    )
    crosswalk = joined[["geoid", "community_area", "community_name"]].drop_duplicates().reset_index(drop=True)

    print(
        f"  Crosswalk: {len(crosswalk)} tracts across "
        f"{crosswalk['community_area'].nunique()} community areas."
    )
    return crosswalk

In [29]:
# STEP 3: Aggregate to community areas + compute SES rates
def safe_rate(n, d):
    d = d.astype("float")
    return n / d.where(d != 0)


def aggregate_to_community_areas(acs_df: pd.DataFrame, crosswalk: pd.DataFrame) -> pd.DataFrame:
    df = acs_df.merge(crosswalk, on="geoid", how="inner")

    # Education derived counts
    lt_hs_cols = [
        "edu_no_schooling", "edu_nursery", "edu_k",
        "edu_g1", "edu_g2", "edu_g3", "edu_g4", "edu_g5",
        "edu_g6", "edu_g7", "edu_g8", "edu_g9", "edu_g10",
        "edu_g11", "edu_g12_no_diploma",
    ]
    df["less_than_hs_count"] = df[lt_hs_cols].sum(axis=1, min_count=1)
    df["bachelor_plus_count"] = df[["edu_bachelors", "edu_masters", "edu_professional", "edu_doctorate"]].sum(axis=1, min_count=1)

    # Rent burden derived count pieces (>=30%)
    df["rent_burden_30p_num"] = df[["rent_30_34", "rent_35_39", "rent_40_49", "rent_50_plus"]].sum(axis=1, min_count=1)
    df["rent_burden_computed_den"] = df["rent_burden_total"] - df["rent_not_computed"]

    # Crowding derived numerator
    df["crowded_num"] = df[
        ["own_crowd_101_150", "own_crowd_151_200", "own_crowd_201_plus",
         "rent_crowd_101_150", "rent_crowd_151_200", "rent_crowd_201_plus"]
    ].sum(axis=1, min_count=1)

    # Aggregate raw counts to community areas
    count_cols = [
        # base pop + labor + poverty + education
        "total_population",
        "below_poverty", "poverty_denom",
        "labor_force", "unemployed",
        "edu_denom", "less_than_hs_count", "bachelor_plus_count",

        # SNAP
        "hh_total", "hh_snap",

        # rent burden + crowding
        "rent_burden_30p_num", "rent_burden_computed_den",
        "rent_50_plus",  # needed for rent_burden_50p_rate
        "occ_hu_total", "crowded_num",

        # NEW: tenure + vacancy
        "occ_hu_tenure_total", "renter_occupied",
        "hu_total", "hu_vacant",

        # NEW: public assistance cash income
        "hh_total_pa", "hh_public_assist",

        # NEW: no-vehicle households
        "hh_total_veh", "hh_no_vehicle",

        # optional context: race/ethnicity
        "race_total", "white_non_hispanic", "black_non_hispanic", "hispanic",
    ]

    missing = [c for c in count_cols if c not in df.columns]
    if missing:
        raise KeyError(f"Missing required ACS columns for aggregation: {missing}")

    ca_counts = df.groupby(["community_area", "community_name"], as_index=False)[count_cols].sum()

    # approximation of tract median income
    df["income_x_pop"] = df["median_household_income"] * df["total_population"]
    income_weighted = df.groupby(["community_area", "community_name"], as_index=False).apply(
        lambda g: pd.Series({
            "median_household_income_weighted": (
                g["income_x_pop"].sum() / g["total_population"].sum()
                if g["total_population"].sum() > 0 else pd.NA
            )
        })
    )
    ca_counts = ca_counts.merge(income_weighted, on=["community_area", "community_name"], how="left")

    # Final SES rates
    ca = ca_counts.copy()

    # Core SES
    ca["poverty_rate"] = safe_rate(ca["below_poverty"], ca["poverty_denom"])
    ca["unemployment_rate"] = safe_rate(ca["unemployed"], ca["labor_force"])
    ca["pct_less_than_hs"] = safe_rate(ca["less_than_hs_count"], ca["edu_denom"])
    ca["pct_bachelor_plus"] = safe_rate(ca["bachelor_plus_count"], ca["edu_denom"])

    ca["snap_rate"] = safe_rate(ca["hh_snap"], ca["hh_total"])
    ca["rent_burden_30p_rate"] = safe_rate(ca["rent_burden_30p_num"], ca["rent_burden_computed_den"])
    ca["crowding_rate"] = safe_rate(ca["crowded_num"], ca["occ_hu_total"])

    # race shares
    ca["pct_white_non_hispanic"] = safe_rate(ca["white_non_hispanic"], ca["race_total"])
    ca["pct_black_non_hispanic"] = safe_rate(ca["black_non_hispanic"], ca["race_total"])
    ca["pct_hispanic"] = safe_rate(ca["hispanic"], ca["race_total"])

    # tenure / vacancy
    ca["pct_renter"] = safe_rate(ca["renter_occupied"], ca["occ_hu_tenure_total"])
    ca["vacancy_rate"] = safe_rate(ca["hu_vacant"], ca["hu_total"])

    # severe rent burden (>=50%) using same denominator
    ca["rent_burden_50p_rate"] = safe_rate(ca["rent_50_plus"], ca["rent_burden_computed_den"])

    # public assistance cash + no vehicle
    ca["public_assist_rate"] = safe_rate(ca["hh_public_assist"], ca["hh_total_pa"])
    ca["no_vehicle_rate"] = safe_rate(ca["hh_no_vehicle"], ca["hh_total_veh"])

    # Round
    rate_cols = [
        "poverty_rate", "unemployment_rate", "pct_less_than_hs", "pct_bachelor_plus",
        "snap_rate", "rent_burden_30p_rate", "rent_burden_50p_rate", "crowding_rate",
        "pct_renter", "vacancy_rate", "public_assist_rate", "no_vehicle_rate",
        "pct_white_non_hispanic", "pct_black_non_hispanic", "pct_hispanic",
    ]
    ca[rate_cols] = ca[rate_cols].round(4)
    ca["median_household_income_weighted"] = ca["median_household_income_weighted"].round(0)

    # Output columns 
    output_cols = [
        "community_area", "community_name", "total_population",
        "poverty_rate", "median_household_income_weighted", "unemployment_rate",
        "pct_less_than_hs", "pct_bachelor_plus",
        "snap_rate", "public_assist_rate",
        "rent_burden_30p_rate", "rent_burden_50p_rate",
        "crowding_rate", "pct_renter", "vacancy_rate", "no_vehicle_rate",
        "pct_white_non_hispanic", "pct_black_non_hispanic", "pct_hispanic",
    ]
    ca = ca[output_cols].sort_values("community_area").reset_index(drop=True)

    # check, 77
    print("  Community areas in output:", ca["community_area"].nunique())
    return ca


In [30]:
def main():
    acs_df = pull_acs_tracts(CENSUS_API_KEY, ACS_YEAR, VAR_NAMES)

    ca_gdf = load_community_area_boundaries()
    tract_gdf = load_tract_boundaries()

    crosswalk = build_crosswalk(tract_gdf, ca_gdf)

    ca_ses = aggregate_to_community_areas(acs_df, crosswalk)

    # Save output
    SAVE_OUTPUT = False  # flip to True when change document

    output_path = "data/acs_community_area_ses_2020_2024.csv"
    if SAVE_OUTPUT:
        ca_ses.to_csv(output_path, index=False)
        print(f"Saved: {output_path}")
    else:
        print("SAVE_OUTPUT=False, not saving file.")

    print(ca_ses.head(10).to_string(index=False))

if __name__ == "__main__":
    main()

  Pulling 45 vars...
  Pulling 10 vars...
  Pulled 1332 Cook County tracts.
  Loaded 77 community areas.
  Loaded 1332 Cook County tracts.
  Crosswalk: 781 tracts across 77 community areas.
  Community areas in output: 77
SAVE_OUTPUT=False, not saving file.
 community_area  community_name  total_population  poverty_rate  median_household_income_weighted  unemployment_rate  pct_less_than_hs  pct_bachelor_plus  snap_rate  public_assist_rate  rent_burden_30p_rate  rent_burden_50p_rate  crowding_rate  pct_renter  vacancy_rate  no_vehicle_rate  pct_white_non_hispanic  pct_black_non_hispanic  pct_hispanic
              1     ROGERS PARK             49628        0.1840                           64218.0             0.0743            0.1219             0.5176     0.1627              0.0446                0.4925                0.2351         0.0516      0.7133        0.0908           0.3604                  0.4194                  0.2312        0.2416
              2      WEST RIDGE             

In [31]:
ses = pd.read_csv("data/acs_community_area_ses_2020_2024.csv")
ses["community_area"].isna().sum()

np.int64(0)

In [32]:
ses["community_area"].nunique()
ses.sort_values("community_area").head(30)

,community_area,community_name,total_population,poverty_rate,median_household_income_weighted,unemployment_rate,pct_less_than_hs,pct_bachelor_plus,snap_rate,public_assist_rate,rent_burden_30p_rate,rent_burden_50p_rate,crowding_rate,pct_renter,vacancy_rate,no_vehicle_rate,pct_white_non_hispanic,pct_black_non_hispanic,pct_hispanic
0,1,ROGERS PARK,49628,0.1840,64218.0,0.0743,0.1219,0.5176,0.1627,0.0446,0.4925,0.2351,0.0516,0.7133,0.0908,0.3604,0.4194,0.2312,0.2416
1,2,WEST RIDGE,78373,0.1929,71554.0,0.0733,0.1279,0.4388,0.2072,0.0353,0.5336,0.2907,0.0843,0.4938,0.0769,0.1499,0.3937,0.1171,0.2058
2,3,UPTOWN,56344,0.1845,74039.0,0.0470,0.0727,0.6087,0.1464,0.0350,0.4687,0.2188,0.0237,0.7072,0.0912,0.4285,0.5197,0.1901,0.1415
3,4,LINCOLN SQUARE,41651,0.0930,91672.0,0.0519,0.0503,0.6781,0.0675,0.0314,0.4328,0.2248,0.0200,0.5928,0.0618,0.2224,0.6232,0.0416,0.1778
4,5,NORTH CENTER,36014,0.0508,148188.0,0.0381,0.0268,0.8049,0.0382,0.0148,0.3052,0.1175,0.0024,0.4313,0.0870,0.1201,0.7334,0.0235,0.1185
5,6,LAKE VIEW,102827,0.0827,120923.0,0.0335,0.0177,0.8326,0.0507,0.0167,0.3884,0.1751,0.0110,0.6249,0.0601,0.3946,0.7289,0.0448,0.1025
6,7,LINCOLN PARK,67987,0.0852,158658.0,0.0431,0.0214,0.8683,0.0357,0.0144,0.3778,0.1606,0.0100,0.5524,0.0992,0.3156,0.7669,0.0247,0.0792
7,8,NEAR NORTH SIDE,100633,0.1014,121327.0,0.0316,0.0188,0.8562,0.0451,0.0161,0.4065,0.1824,0.0230,0.6390,0.0975,0.4768,0.6495,0.0838,0.0808
8,9,EDISON PARK,11558,0.0812,130037.0,0.0337,0.0219,0.5751,0.0254,0.0077,0.4829,0.2579,0.0150,0.1880,0.0549,0.0560,0.7943,0.0061,0.1492
9,10,NORWOOD PARK,39530,0.0566,110250.0,0.0437,0.0647,0.4752,0.0720,0.0241,0.4628,0.2741,0.0138,0.1909,0.0378,0.0712,0.7061,0.0095,0.1801


In [33]:
ses.columns

Index(['community_area', 'community_name', 'total_population', 'poverty_rate',
       'median_household_income_weighted', 'unemployment_rate',
       'pct_less_than_hs', 'pct_bachelor_plus', 'snap_rate',
       'public_assist_rate', 'rent_burden_30p_rate', 'rent_burden_50p_rate',
       'crowding_rate', 'pct_renter', 'vacancy_rate', 'no_vehicle_rate',
       'pct_white_non_hispanic', 'pct_black_non_hispanic', 'pct_hispanic'],
      dtype='str')

### Merging 311 and ACS data

In [34]:

acs = pd.read_csv("data/acs_community_area_ses_2020_2024.csv")
m311 = pd.read_csv("data/311_metrics.csv")

for c in ["community_area","year","ward","n_requests","n_closed","share_closed","median_ttc_hours","p75_ttc_hours"]:
    if c in m311.columns:
        m311[c] = pd.to_numeric(m311[c], errors="coerce")

m311_ca = (
    m311.dropna(subset=["community_area","n_requests"])
        .groupby("community_area", as_index=False)
        .apply(lambda g: pd.Series({
            "n_requests_total": g["n_requests"].sum(),
            "share_closed_overall": g["n_closed"].sum() / g["n_requests"].sum() if g["n_requests"].sum() else np.nan,
            "median_ttc_wavg": np.average(g["median_ttc_hours"], weights=g["n_requests"]) if g["median_ttc_hours"].notna().any() else np.nan,
            "p75_ttc_wavg": np.average(g["p75_ttc_hours"], weights=g["n_requests"]) if g["p75_ttc_hours"].notna().any() else np.nan,
        }))
        .reset_index(drop=True)
)

master_ca = acs.merge(m311_ca, on="community_area", how="left")
# master_ca.to_csv("data/master_acs_311_ca.csv", index=False) 

master_ca.head()


,community_area,community_name,total_population,poverty_rate,median_household_income_weighted,unemployment_rate,pct_less_than_hs,pct_bachelor_plus,snap_rate,public_assist_rate,...,pct_renter,vacancy_rate,no_vehicle_rate,pct_white_non_hispanic,pct_black_non_hispanic,pct_hispanic,n_requests_total,share_closed_overall,median_ttc_wavg,p75_ttc_wavg
0,1,ROGERS PARK,49628,0.1840,64218.0,0.0743,0.1219,0.5176,0.1627,0.0446,...,0.7133,0.0908,0.3604,0.4194,0.2312,0.2416,49646.0,0.967530,46.121575,303.168290
1,2,WEST RIDGE,78373,0.1929,71554.0,0.0733,0.1279,0.4388,0.2072,0.0353,...,0.4938,0.0769,0.1499,0.3937,0.1171,0.2058,75476.0,0.968798,69.134867,416.640807
2,3,UPTOWN,56344,0.1845,74039.0,0.0470,0.0727,0.6087,0.1464,0.0350,...,0.7072,0.0912,0.4285,0.5197,0.1901,0.1415,56086.0,0.964608,45.225411,293.130706
3,4,LINCOLN SQUARE,41651,0.0930,91672.0,0.0519,0.0503,0.6781,0.0675,0.0314,...,0.5928,0.0618,0.2224,0.6232,0.0416,0.1778,53312.0,0.965955,88.820217,370.281342
4,5,NORTH CENTER,36014,0.0508,148188.0,0.0381,0.0268,0.8049,0.0382,0.0148,...,0.4313,0.0870,0.1201,0.7334,0.0235,0.1185,53299.0,0.963658,62.041159,422.072186
